### Quét các thiết bị Bluetooth

In [5]:
import asyncio
from bleak import BleakScanner


async def main():
    print("Scanning for BLE devices...\n")

    devices = await BleakScanner.discover(timeout=10)

    for device in devices:
        print(device)


await main()

Scanning for BLE devices...

F9F593C1-63AF-F843-0655-F003ED36DD2C: None
3AA6C6FE-BCB8-54FC-7D29-194243B35024: Google Pixel 9
78A64D2E-9DAB-FD75-3E4F-BEDBBE5C4059: S503_f2766d
83EF2FBE-4143-A627-1371-A68D52C6499A: None
4600F1A4-1AE6-7C9D-5B14-46B26BA84280: None
043D7944-1B3D-EBD2-1CCD-1225FBE9E0ED: HAVIT LIFE NC02H
8E13781C-552F-E2B7-F08C-82EA6DAB723B: None
AE985D3E-A639-2FE8-53BE-4C648B07D089: None
512CE026-FAC7-3657-1E4C-F091E7C7AA0B: GBeacon
04215D9D-7FAC-5E68-2190-344EC7EE2A27: None
14FE0FB6-500D-2645-8862-704E16BB787F: None
6030F388-154E-B786-97B8-848E980206B5: None
4D2A6A36-391A-E39D-626C-D6A1C2814C77: None
54B9253D-4535-5262-F290-FA9E8959173C: None
18EFFEC8-E6B4-7406-C9A2-4689D74AD4A1: MoMo
135D195F-AFFA-7F14-1384-87B8A23BF004: None
BE265D31-44D1-85B5-8F57-D089388B5C9E: PHUQIBOZIEN
C31D3E66-68F4-5505-1F9B-8B6CC40A42DD: None
05B2C50A-84B8-9A49-6C14-77979B7F1626: None
43F02C54-BC66-9063-EE2E-902241BFD96C: MoMo
7F7E7610-F8EF-63FB-16B3-ED2CD724C6EA: None


### Kết nối với IMU và thu vào raw_result (100Hz)

In [3]:
import asyncio
from bleak import BleakClient

H10_ADDRESS = "62780A7A-0F91-CA11-5534-7541D13CA933"

PMD_CP_UUID = "FB005C81-02E7-F387-1CAD-8ACD2D8DF0C8"
PMD_DATA_UUID = "FB005C82-02E7-F387-1CAD-8ACD2D8DF0C8"

DURATION_SECONDS = 10

# Lưu toàn bộ packet RAW
raw_result = []


def control_handler(sender, data):
    print("PMD CONTROL:", data.hex(" "))


def data_handler(sender, data):
    # Không decode ở đây
    raw_result.append(bytes(data))


async def collect_raw():

    raw_result.clear()

    async with BleakClient(
        H10_ADDRESS,
        timeout=15
    ) as client:

        print("Connected:", client.is_connected)

        # Enable notifications
        await client.start_notify(
            PMD_CP_UUID,
            control_handler
        )

        await client.start_notify(
            PMD_DATA_UUID,
            data_handler
        )

        print("PMD notifications enabled.")

        # -----------------------------------------
        # ACC 200 Hz / 16-bit / 8G
        # -----------------------------------------

        ACC_START = bytes([
            0x02,
            0x02,

            0x00, 0x01, 0x64,        # 100 Hz
            0x00, 0x01, 0x01, 0x10,  # 16-bit
            0x00, 0x02, 0x01, 0x08,  # 8G
            0x00
        ])

        ACC_STOP = bytes([
            0x03,
            0x02
        ])

        print("Starting ACC...")
        print("Command:", ACC_START.hex(" "))

        await client.write_gatt_char(
            PMD_CP_UUID,
            ACC_START,
            response=True
        )

        print("Collecting RAW data...")

        await asyncio.sleep(DURATION_SECONDS)

        print("Stopping ACC...")

        try:
            await client.write_gatt_char(
                PMD_CP_UUID,
                ACC_STOP,
                response=True
            )
        except Exception as e:
            print("STOP error:", e)

        await client.stop_notify(PMD_DATA_UUID)
        await client.stop_notify(PMD_CP_UUID)

    print("\nFinished.")
    print("Number of RAW packets:", len(raw_result))


await collect_raw()

Connected: True
PMD notifications enabled.
Starting ACC...
Command: 02 02 00 01 64 00 01 01 10 00 02 01 08 00
PMD CONTROL: f0 02 02 00 00 01
Stopping ACC...
PMD CONTROL: f0 03 02 00 00

Finished.
Number of RAW packets: 27


In [4]:
import struct
import pandas as pd
import numpy as np


# ============================================================
# CONFIG
# ============================================================

SAMPLE_RATE = 100.0
ACC_RANGE_G = 8.0
ACC_SCALE = ACC_RANGE_G / 32768.0

decoded_result = []


# ============================================================
# DECODE 1 PACKET
# ============================================================

def decode_acc_packet(packet):

    if len(packet) < 16:
        return []

    # ACC
    if packet[0] != 0x02:
        return []

    # --------------------------------------------------------
    # Header
    # --------------------------------------------------------

    timestamp_ns = int.from_bytes(
        packet[1:9],
        byteorder="little",
        signed=False
    )

    timestamp_s = timestamp_ns / 1e9

    # Byte 9
    frame_type = packet[9]

    # --------------------------------------------------------
    # ACC DATA
    #
    # Packet 226 bytes:
    #
    # 10-byte header
    # + 36 samples × 6 bytes
    #
    # 10 + 36*6 = 226
    # --------------------------------------------------------

    offset = 10

    samples = []

    sample_index = 0

    while offset + 6 <= len(packet):

        raw_x, raw_y, raw_z = struct.unpack_from(
            "<hhh",
            packet,
            offset
        )

        acc_x = raw_x * ACC_SCALE
        acc_y = raw_y * ACC_SCALE
        acc_z = raw_z * ACC_SCALE

        sample_timestamp = (
            timestamp_s +
            sample_index / SAMPLE_RATE
        )

        samples.append({
            "TimeStamp(s)": sample_timestamp,
            "AccX": acc_x,
            "AccY": acc_y,
            "AccZ": acc_z
        })

        offset += 6
        sample_index += 1

    return samples


# ============================================================
# DECODE ALL RAW PACKETS
# ============================================================

decoded_result.clear()

packet_info = []

for packet_index, packet in enumerate(raw_result):

    samples = decode_acc_packet(packet)

    decoded_result.extend(samples)

    packet_info.append({
        "Packet": packet_index,
        "Length": len(packet),
        "Samples": len(samples)
    })


# ============================================================
# DATAFRAME
# ============================================================

df_acc = pd.DataFrame(decoded_result)


# ============================================================
# BASIC INFORMATION
# ============================================================

print("=" * 60)
print("RAW DATA")
print("=" * 60)

print("Số RAW packets :", len(raw_result))

if raw_result:
    print("Packet đầu tiên :", len(raw_result[0]), "bytes")


print("\n" + "=" * 60)
print("DECODED DATA")
print("=" * 60)

print("Số samples :", len(df_acc))


# ============================================================
# PACKET INFORMATION
# ============================================================

df_packet_info = pd.DataFrame(packet_info)

print("\n" + "=" * 60)
print("PACKET INFORMATION")
print("=" * 60)

print(df_packet_info.head(10))

print("\nSamples / packet:")
print(
    df_packet_info["Samples"]
    .value_counts()
    .sort_index()
)


# ============================================================
# FIRST SAMPLES
# ============================================================

print("\n" + "=" * 60)
print("20 SAMPLES ĐẦU")
print("=" * 60)

print(
    df_acc.head(20).to_string(index=False)
)


# ============================================================
# ACC STATISTICS
# ============================================================

print("\n" + "=" * 60)
print("ACC STATISTICS")
print("=" * 60)

print(
    df_acc[["AccX", "AccY", "AccZ"]].describe()
)


# ============================================================
# ACC MAGNITUDE
# ============================================================

df_acc["AccMagnitude"] = np.sqrt(
    df_acc["AccX"] ** 2 +
    df_acc["AccY"] ** 2 +
    df_acc["AccZ"] ** 2
)


print("\n" + "=" * 60)
print("ACC MAGNITUDE")
print("=" * 60)

print(
    df_acc["AccMagnitude"].describe()
)


# ============================================================
# TIMESTAMP / SAMPLE RATE
# ============================================================

if len(df_acc) > 1:

    dt = df_acc["TimeStamp(s)"].diff().dropna()

    print("\n" + "=" * 60)
    print("TIMESTAMP / SAMPLE RATE")
    print("=" * 60)

    print("dt mean     :", dt.mean(), "s")
    print("Sample rate :", 1 / dt.mean(), "Hz")

    print("\nFirst 20 dt:")
    print(
        dt.head(20).to_string(index=False)
    )


# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("VALIDATION")
print("=" * 60)

print(
    "NaN:",
    df_acc[["AccX", "AccY", "AccZ"]]
    .isna()
    .sum()
    .sum()
)

print(
    "AccX range:",
    df_acc["AccX"].min(),
    "→",
    df_acc["AccX"].max()
)

print(
    "AccY range:",
    df_acc["AccY"].min(),
    "→",
    df_acc["AccY"].max()
)

print(
    "AccZ range:",
    df_acc["AccZ"].min(),
    "→",
    df_acc["AccZ"].max()
)

print(
    "Magnitude mean:",
    df_acc["AccMagnitude"].mean(),
    "G"
)

print(
    "Magnitude median:",
    df_acc["AccMagnitude"].median(),
    "G"
)

print(
    "Magnitude max:",
    df_acc["AccMagnitude"].max(),
    "G"
)

RAW DATA
Số RAW packets : 27
Packet đầu tiên : 226 bytes

DECODED DATA
Số samples : 972

PACKET INFORMATION
   Packet  Length  Samples
0       0     226       36
1       1     226       36
2       2     226       36
3       3     226       36
4       4     226       36
5       5     226       36
6       6     226       36
7       7     226       36
8       8     226       36
9       9     226       36

Samples / packet:
Samples
36    27
Name: count, dtype: int64

20 SAMPLES ĐẦU
 TimeStamp(s)      AccX     AccY     AccZ
 5.996161e+08 -0.191162 0.046143 0.160400
 5.996161e+08 -0.190918 0.046143 0.159424
 5.996161e+08 -0.190918 0.046875 0.160156
 5.996161e+08 -0.190674 0.046875 0.160400
 5.996161e+08 -0.190430 0.047607 0.158691
 5.996161e+08 -0.190430 0.047607 0.157227
 5.996161e+08 -0.190430 0.048096 0.157471
 5.996161e+08 -0.190674 0.047607 0.158447
 5.996161e+08 -0.190186 0.048340 0.159912
 5.996161e+08 -0.190674 0.048096 0.161865
 5.996161e+08 -0.190918 0.047852 0.162842
 5.996161e+08